In [4]:
resp_api = requests.get(API_URL, params=PAYLOAD)
print("Status code:", resp_api.status_code)
print("Respuesta raw:", resp_api.text[:500])

Status code: 200
Respuesta raw: {
  "filtros": {
    "cambiador": false,
    "carrito": true,
    "edad_max": "3",
    "lluvia": true,
    "mascotas": false,
    "silla_ruedas": false,
    "ubicacion": "Bilbao"
  },
  "resultados": [
    {
      "categoria": "restauracion",
      "descripcion": "Este moderno y acogedor restaurante fue dise\u00f1ado por el arquitecto Ricardo Legorreta y est...",
      "direccion": "Lehendakari Leizaola, 29. (H. Meli\u00e1 Bilbao)",
      "edad_minima": 0,
      "email": "aizian@restaurante-aizi


In [6]:
import requests
import numpy as np

# ── Configuración ──────────────────────────────────────────────
API_URL    = "http://127.0.0.1:5000/planes"
OLLAMA_URL = "http://localhost:11434/api/embeddings"
MODELOS    = ["qwen3-embedding:0.6b", "embeddinggemma:latest", "nomic-embed-text:latest"]

PAYLOAD = {
    "ubicacion":    "Bilbao",
    "edad_max":     3,
    "lluvia":       True,
    "carrito":      True,
    "cambiador":    False,
    "silla_ruedas": False,
    "mascotas":     False,
    "limite":       20
}

# ── Texto de consulta generado desde los parámetros ────────────
CONSULTA = "planes para niños de hasta 3 años en Bilbao, con lluvia, accesible con carrito"

# ── Helpers ────────────────────────────────────────────────────
def get_embedding(texto, modelo):
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={"model": modelo, "prompt": texto},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()["embedding"]
    except Exception as e:
        print(f"ERROR embedding: {e}")
        return None

def cosine_similarity(v1, v2):
    a, b = np.array(v1), np.array(v2)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

def construir_texto_lugar(lugar):
    partes = [
        str(lugar.get("nombre",         "") or ""),
        str(lugar.get("categoria",       "") or ""),
        str(lugar.get("tipo_plantilla",  "") or ""),
        str(lugar.get("descripcion",     "") or ""),
        str(lugar.get("municipio",       "") or ""),
    ]
    return ". ".join(p.strip() for p in partes if p.strip())

# ── 1. Llamar a la API ─────────────────────────────────────────
print("Llamando a /planes...")
resp_api = requests.get(API_URL, params=PAYLOAD)
resultados = resp_api.json()

# Ajusta esta línea si tu API devuelve {"planes": [...]} u otra estructura
if isinstance(resultados, dict):
    resultados = resultados.get("resultados")

print(f"Resultados recibidos: {len(resultados)}\n")

# ── 2. Comparar modelos ────────────────────────────────────────
for modelo in MODELOS:
    print(f"\n{'='*60}")
    print(f"  MODELO: {modelo}")
    print(f"{'='*60}")

    emb_consulta = get_embedding(CONSULTA, modelo)
    if emb_consulta is None:
        print("  ERROR: no se pudo vectorizar la consulta")
        continue

    scores = []
    for lugar in resultados:
        texto  = construir_texto_lugar(lugar)
        emb    = get_embedding(texto, modelo)
        if emb is None:
            continue
        score = cosine_similarity(emb_consulta, emb)
        scores.append((lugar.get("nombre", "sin nombre"), score))

    # Ordenar por similitud descendente
    scores.sort(key=lambda x: x[1], reverse=True)

    print(f"\n  Ranking reordenado por similitud coseno:\n")
    for i, (nombre, score) in enumerate(scores, 1):
        barra = "█" * int(score * 20)
        print(f"  {i:>2}. {score:.3f} {barra:<20} {nombre}")

Llamando a /planes...
Resultados recibidos: 20


  MODELO: qwen3-embedding:0.6b

  Ranking reordenado por similitud coseno:

   1. 0.368 ███████              Serantes
   2. 0.363 ███████              Víctor
   3. 0.344 ██████               La Escuela
   4. 0.322 ██████               La Gabarra
   5. 0.313 ██████               Txakoli Simon
   6. 0.304 ██████               Le Bol Blanc
   7. 0.302 ██████               Atelier Etxanobe
   8. 0.296 █████                Café Iruña
   9. 0.289 █████                Kate Zaharra
  10. 0.288 █████                Beraia
  11. 0.282 █████                Cafetería Abandoibarra
  12. 0.281 █████                Mina
  13. 0.280 █████                Nerua
  14. 0.279 █████                5ª Planta
  15. 0.276 █████                Dena-Ona
  16. 0.273 █████                Beltz
  17. 0.264 █████                Aizian
  18. 0.255 █████                Arraiz
  19. 0.237 ████                 Etxaniz
  20. 0.229 ████                 Serantes II

  MODELO

In [7]:
import pandas as pd
from pathlib import Path

# Ajusta esta ruta a donde tienes los CSV
CARPETA = r"C:\\Users\\rns_2\\Documents\\Desafio-Data\\recomendador\\datos_ode_enriquecidos"

CSV_FILES = [
    "centros-comerciales_2026-06-02.csv",
    "comercios_ode_completo_2026-06-02.csv",
    "espacios-naturales_2026-06-02.csv",
    "kulturklik_2026-06-02.csv",
    "museos_2026-06-02.csv",
    "ocio_2026-06-02.csv",
    "planes_2026-06-02.csv",
    "recursos-culturales_2026-06-02.csv",
    "restaurantes_2026-06-02.csv",
    "rutas_2026-06-02.csv",
]

dfs = []
for filename in CSV_FILES:
    path = Path(CARPETA) / filename
    try:
        df = pd.read_csv(path, encoding="utf-8")
        df["fuente_archivo"] = filename  # para saber de dónde viene cada registro
        dfs.append(df)
        print(f"  ✓ {filename:<45} {len(df):>5} registros")
    except Exception as e:
        print(f"  ✗ {filename} — ERROR: {e}")

pool = pd.concat(dfs, ignore_index=True)

print(f"\nTotal registros en el pool : {len(pool)}")
print(f"Columnas                   : {pool.columns.tolist()}")

  ✓ centros-comerciales_2026-06-02.csv               19 registros
  ✓ comercios_ode_completo_2026-06-02.csv          1827 registros
  ✓ espacios-naturales_2026-06-02.csv                84 registros
  ✓ kulturklik_2026-06-02.csv                      2447 registros
  ✓ museos_2026-06-02.csv                           151 registros
  ✓ ocio_2026-06-02.csv                              17 registros
  ✓ planes_2026-06-02.csv                           340 registros
  ✓ recursos-culturales_2026-06-02.csv              650 registros
  ✓ restaurantes_2026-06-02.csv                     603 registros
  ✓ rutas_2026-06-02.csv                            114 registros

Total registros en el pool : 6252
Columnas                   : ['fuente', 'external_id', 'nombre', 'descripcion', 'categoria', 'tipo_plantilla', 'municipio', 'territorio', 'direccion', 'lat', 'lng', 'telefono', 'email', 'website', 'es_lluvia', 'es_carrito', 'es_cambiador', 'es_silla_ruedas', 'es_mascotas', 'edad_minima', 'fuente_archivo'

In [10]:
def construir_texto(row):
    partes = [
        str(row.get("nombre",         "") or ""),
        str(row.get("categoria",      "") or ""),
        str(row.get("tipo_plantilla", "") or ""),
        str(row.get("descripcion",    "") or ""),
        str(row.get("municipio",      "") or ""),
        str(row.get("territorio",     "") or ""),
    ]
    return ". ".join(p.strip() for p in partes if p.strip())

In [11]:
# Muestra estratificada — proporcional por fuente_archivo
MUESTRA_POR_FUENTE = 10  # 10 de cada CSV = 100 registros variados

muestra_pool = (
    pool
    .dropna(subset=["nombre"])
    .groupby("fuente_archivo", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MUESTRA_POR_FUENTE), random_state=42))
    .reset_index(drop=True)
)

# Construir texto de embedding para cada registro
muestra_pool["texto_embedding"] = muestra_pool.apply(construir_texto, axis=1)

print(f"Total muestra : {len(muestra_pool)}")
print(f"\nDistribución por fuente:")
print(muestra_pool["fuente_archivo"].value_counts().to_string())
print(f"\nEjemplos de textos generados:")
for _, row in muestra_pool.head(3).iterrows():
    print(f"\n  [{row['fuente_archivo']}]")
    print(f"  {row['texto_embedding'][:120]}")

Total muestra : 100

Distribución por fuente:
fuente_archivo
centros-comerciales_2026-06-02.csv       10
comercios_ode_completo_2026-06-02.csv    10
espacios-naturales_2026-06-02.csv        10
kulturklik_2026-06-02.csv                10
museos_2026-06-02.csv                    10
ocio_2026-06-02.csv                      10
planes_2026-06-02.csv                    10
recursos-culturales_2026-06-02.csv       10
restaurantes_2026-06-02.csv              10
rutas_2026-06-02.csv                     10

Ejemplos de textos generados:

  [centros-comerciales_2026-06-02.csv]
  Dendaraba. comercio. Centros comerciales. nan. Vitoria-Gasteiz. araba

  [centros-comerciales_2026-06-02.csv]
  Bidarte. comercio. Centros comerciales. nan. Bilbao. bizkaia

  [centros-comerciales_2026-06-02.csv]
  Garbera. comercio. Centros comerciales. nan. Donostia / San Sebastián. gipuzkoa


C:\Users\rns_2\AppData\Local\Temp\ipykernel_8136\3138872095.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), MUESTRA_POR_FUENTE), random_state=42))


In [15]:
CONSULTA = "planes para niños de hasta 3 años en Bilbao, de interior, accesible con carrito"
MODELOS  = ["qwen3-embedding:0.6b", "embeddinggemma:latest"]
TOP_K    = 10  # mostrar top 10 de cada modelo

for modelo in MODELOS:
    print(f"\n{'='*60}")
    print(f"  MODELO: {modelo}")
    print(f"{'='*60}")

    emb_consulta = get_embedding(CONSULTA, modelo)
    if emb_consulta is None:
        print("  ERROR: no se pudo vectorizar la consulta")
        continue

    scores = []
    for _, row in muestra_pool.iterrows():
        texto = row["texto_embedding"]
        emb   = get_embedding(texto, modelo)
        if emb is None:
            continue
        score = cosine_similarity(emb_consulta, emb)
        scores.append({
            "nombre":   row.get("nombre", "sin nombre"),
            "categoria": row.get("categoria", ""),
            "municipio": row.get("municipio", ""),
            "fuente":   row["fuente_archivo"].replace("_2026-06-02.csv", ""),
            "score":    score,
        })

    scores.sort(key=lambda x: x["score"], reverse=True)

    print(f"\n  Top {TOP_K} resultados para: '{CONSULTA}'\n")
    for i, r in enumerate(scores[:TOP_K], 1):
        barra = "█" * int(r["score"] * 20)
        print(f"  {i:>2}. {r['score']:.3f} {barra:<15} [{r['fuente']:<22}] {r['nombre']} ({r['municipio']})")

    print(f"\n  Distribución de scores:")
    all_scores = [r["score"] for r in scores]
    print(f"  Media : {np.mean(all_scores):.3f}")
    print(f"  Std   : {np.std(all_scores):.3f}")
    print(f"  Min   : {np.min(all_scores):.3f}")
    print(f"  Max   : {np.max(all_scores):.3f}")


  MODELO: qwen3-embedding:0.6b

  Top 10 resultados para: 'planes para niños de hasta 3 años en Bilbao, de interior, accesible con carrito'

   1. 0.503 ██████████      [centros-comerciales   ] Bidarte (Bilbao)
   2. 0.488 █████████       [ocio                  ] De pintxos por el Ensanche de Bilbao (Bilbao)
   3. 0.456 █████████       [ocio                  ] De pintxos por el Casco Viejo de Bilbao (Bilbao)
   4. 0.411 ████████        [kulturklik            ] Aste Nagusia de Bilbao 2026: Jordi Merca: "Yo sobreviví a la EGB" (Bilbao)
   5. 0.400 ████████        [ocio                  ] Mendexa Abentura Park (Mendexa)
   6. 0.394 ███████         [centros-comerciales   ] Bilbondo (Basauri)
   7. 0.385 ███████         [planes                ] Parque de aventura en Otxandio (Otxandio)
   8. 0.382 ███████         [kulturklik            ] "Euskal Herria Zuzenean" (Bilbao)
   9. 0.373 ███████         [centros-comerciales   ] Ballonti (Portugalete)
  10. 0.364 ███████         [planes         

============================================================
  MODELO: nomic-embed-text:latest
============================================================

  Top 10 resultados para: 'planes para niños de hasta 3 años en Bilbao, de interior, accesible con carrito'

   1. 0.724 ██████████████  [planes                ] Senderismo para todos en las montañas más bonitas de Gipuzkoa (Donostia / San Sebastián)
   2. 0.697 █████████████   [espacios-naturales    ] Playa de Kanalape (Gautegiz Arteaga)
   3. 0.683 █████████████   [comercios_ode_completo] Playa de Itzurun (Zumaia)
   4. 0.677 █████████████   [planes                ] Parque de aventura en Otxandio (Otxandio)
   5. 0.672 █████████████   [espacios-naturales    ] Playa de Ereaga (Getxo)
   6. 0.671 █████████████   [rutas                 ] Ruta del Queso Idiazábal  GR 283 (Segura)
   7. 0.670 █████████████   [espacios-naturales    ] Salburua (Vitoria-Gasteiz)
   8. 0.669 █████████████   [restaurantes          ] Artzai-Enea (Beasain)
   9. 0.669 █████████████   [espacios-naturales    ] Piscinas fluviales de Fresnedo (Campezo/Kanpezu)
  10. 0.667 █████████████   [ocio                  ] De pintxos por Gros, San Sebastián (Donostia / San Sebastián)

  Distribución de scores:
  Media : 0.599
  Std   : 0.065
  Min   : 0.358
  Max   : 0.724